# 02 Systematic search and screening of the defence papers

Design log.
- Search strings are committed first (`schema/search_queries.yaml`, the strings of plan v2 A5 plus the word forms defenses and defending, because the arXiv index does not document stemming) and then run: arXiv by its API (abstract field, cs.CR/cs.CL/cs.LG/cs.AI, submitted from 1 January 2023 to the run date), OpenAlex by its API (title-and-abstract search, which covers IEEE, ACM, Springer, Elsevier, MDPI and ACL-indexed articles), DBLP by its API (titles of conference and journal papers). Amendment to plan v2 A5: these two APIs replace browser exports as the route to the publisher-indexed venues; exported RIS, BibTeX or CSV files dropped into `data/search_exports/<source>/` are still imported and logged if present.
- Citation searching: the seed defences named in plan A5 are resolved on arXiv by name together with "prompt injection"; every arXiv identifier from 2023 onwards printed in the thirteen citation-source PDFs (the nine closest reviews, SecFid, Jia, Nasr, Li, Deep) is collected and its metadata fetched. Both are logged under their own source names so the PRISMA flow can show them.
- OpenAlex has required a free API key since February 2026; the key is read from `.secrets/openalex_key.txt` in the Drive project folder (asked for once, by getpass, if missing) and is never written to the repository or the log. Without it each OpenAlex query returns its first page only and the log says so. DBLP failures are logged with the start of the response so they can be diagnosed.
- Every query is logged with the exact string as sent, the fields, the filters, the date, the count and the file of returned identifiers (`reports/search_log.csv`, `reports/search_raw/`).
- Records are merged by arXiv id, then DOI, then normalised title; a preprint and its published version are linked as one study (`linked_to`). The merged table with abstracts is `reports/screening.csv`.
- Screening is a separate, committed file, `schema/screening_decisions.csv` (key, decision include/exclude/unsure, one reason from the fixed list for an exclusion, access status, note). Fixed reasons: detector only; system level; attack only; benchmark only; jailbreak or direct injection only; survey, position or other; duplicate version; out of period; no full text obtainable; not about prompt injection. Re-measurement studies of published defences are included with the note `role: re-measurement`, since the plan records who measured each evaluation. On the first run the file does not exist and every record is `pending`; the gate stays open. When the decisions file is in place, rerunning this notebook applies it, writes the PRISMA counts, builds the defence manifest for included and unsure records and downloads their arXiv PDFs into `data/defences/` (not committed). That second run is the corpus-freeze commit.
- Outside Colab, `SOURCEMAP_OFFLINE=1` skips the network; the parsers and the merge are covered by `tests/test_search.py`.
- CPU; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for module, package in (("yaml", "pyyaml"), ("pypdf", "pypdf")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd())

## Stage 0b. OpenAlex API key (free; kept outside the repository)

In [ ]:
# OpenAlex has required a free API key since February 2026 (openalex.org, Settings > API key). The key lives in the
# Drive project folder, next to .git-credentials, never inside the repository, and is never written to any log.
import getpass
KEY_FILE = Path(PROJECT_ROOT) / ".secrets" / "openalex_key.txt"
OPENALEX_KEY = os.environ.get("OPENALEX_API_KEY", "")
if not OPENALEX_KEY and KEY_FILE.exists():
    OPENALEX_KEY = KEY_FILE.read_text().strip()
IN_COLAB = "google.colab" in sys.modules
if not OPENALEX_KEY and IN_COLAB:
    entered = getpass.getpass("OpenAlex API key (paste here; Enter to skip and take first pages only): ").strip()
    if entered:
        KEY_FILE.parent.mkdir(parents=True, exist_ok=True)
        KEY_FILE.write_text(entered + "\n")
        OPENALEX_KEY = entered
print("OpenAlex key:", "present" if OPENALEX_KEY else "absent (first page of each OpenAlex query only)")

## Stage 1. Search module, tests and the committed search strings

In [ ]:
%%writefile src/search.py
"""Systematic search for defence papers: arXiv, OpenAlex and DBLP by API, exported record files, citation lists."""
from __future__ import annotations

import csv
import json
import os
import re
import time
import unicodedata
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path

ARXIV_API = os.environ.get("SOURCEMAP_ARXIV_API", "https://export.arxiv.org/api/query")      # overridable for dry runs
OPENALEX_API = os.environ.get("SOURCEMAP_OPENALEX_API", "https://api.openalex.org/works")
DBLP_API = os.environ.get("SOURCEMAP_DBLP_API", "https://dblp.org/search/publ/api")
ATOM = {"a": "http://www.w3.org/2005/Atom", "arxiv": "http://arxiv.org/schemas/atom", "os": "http://a9.com/-/spec/opensearch/1.1/"}
USER_AGENT = "Mozilla/5.0 (compatible; sourcemap-research literature search; +https://github.com/anasbiswas1/sourcemap-research)"
OFFLINE = os.environ.get("SOURCEMAP_OFFLINE") == "1"
GAP = 3.0                      # seconds between requests to one service
ARXIV_CATS = ["cs.CR", "cs.CL", "cs.LG", "cs.AI"]
ARXIV_ID_RE = re.compile(r"(?<![\d.])((?:2[3-6])(?:0[1-9]|1[0-2])\.\d{4,5})(?:v\d+)?(?!\d)")   # 2301.xxxxx to 2612.xxxxx

EXCLUSION_REASONS = ["detector only", "system level", "attack only", "benchmark only", "jailbreak or direct injection only",
                     "survey, position or other", "duplicate version", "out of period", "no full text obtainable",
                     "not about prompt injection"]
DECISIONS = {"include", "exclude", "unsure"}

RECORD_FIELDS = ["key", "title", "first_author", "year", "date", "venue", "arxiv", "arxiv_version", "doi", "abstract",
                 "sources", "queries", "n_hits", "linked_to"]
SCREEN_FIELDS = RECORD_FIELDS + ["decision", "reason", "access_status", "note"]
LOG_FIELDS = ["date", "source", "query_id", "query_as_sent", "fields", "filters", "count", "status", "file"]


# ---- small helpers ------------------------------------------------------------------------------------------

def norm_title(title: str) -> str:
    s = unicodedata.normalize("NFKC", title or "").lower()
    return re.sub(r"[^a-z0-9]+", " ", s).strip()


def norm_key(title: str) -> str:
    return re.sub(r"\s+", "", norm_title(title))[:90]


def _get(url: str, timeout: int = 90) -> bytes:
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT, "Accept": "application/json, application/atom+xml, */*"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read()


def read_csv(path: Path) -> list[dict]:
    path = Path(path)
    if not path.exists():
        return []
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def write_csv(path: Path, rows: list[dict], fieldnames: list[str]) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow({k: ("" if r.get(k) is None else r.get(k)) for k in fieldnames})


def arxiv_id_from_text(s: str) -> str:
    m = ARXIV_ID_RE.search(s or "")
    return m.group(1) if m else ""


# ---- arXiv --------------------------------------------------------------------------------------------------

def arxiv_query_string(must: list[str], date_from: str, date_to: str, cats: list[str] = ARXIV_CATS, field: str = "abs") -> str:
    """must: phrases or words that must all appear in the abstract; dates as YYYYMMDD."""
    parts = []
    for term in must:
        term = term.strip()
        parts.append(f'{field}:"{term}"' if " " in term or "-" in term else f"{field}:{term}")
    cat = " OR ".join(f"cat:{c}" for c in cats)
    return " AND ".join(parts) + f" AND ({cat}) AND submittedDate:[{date_from}0000 TO {date_to}2359]"


def parse_arxiv_feed(xml_bytes: bytes) -> tuple[list[dict], int]:
    """Records of an arXiv API feed and the feed's total-results count."""
    root = ET.fromstring(xml_bytes)
    total = int(root.findtext("os:totalResults", default="0", namespaces=ATOM) or 0)
    out = []
    for e in root.findall("a:entry", ATOM):
        full = (e.findtext("a:id", default="", namespaces=ATOM) or "").rsplit("/", 1)[-1]
        m = re.match(r"(.+?)v(\d+)$", full)
        base, version = (m.group(1), int(m.group(2))) if m else (full, "")
        if not base:
            continue
        authors = [a.findtext("a:name", default="", namespaces=ATOM) for a in e.findall("a:author", ATOM)]
        out.append({
            "arxiv": base, "arxiv_version": version,
            "title": " ".join((e.findtext("a:title", default="", namespaces=ATOM) or "").split()),
            "abstract": " ".join((e.findtext("a:summary", default="", namespaces=ATOM) or "").split()),
            "date": (e.findtext("a:published", default="", namespaces=ATOM) or "")[:10],
            "updated": (e.findtext("a:updated", default="", namespaces=ATOM) or "")[:10],
            "first_author": authors[0] if authors else "", "authors": authors,
            "primary_category": (e.find("arxiv:primary_category", ATOM).get("term") if e.find("arxiv:primary_category", ATOM) is not None else ""),
            "journal_ref": e.findtext("arxiv:journal_ref", default="", namespaces=ATOM) or "",
            "doi": (e.findtext("arxiv:doi", default="", namespaces=ATOM) or "").lower(),
            "venue": e.findtext("arxiv:journal_ref", default="", namespaces=ATOM) or "arXiv",
        })
    return out, total


def arxiv_search(query: str, max_total: int = 3000, page: int = 200, fetch=None) -> tuple[list[dict], str]:
    """All records for an arXiv API search_query, paged; returns (records, status)."""
    if OFFLINE and fetch is None:
        return [], "offline"
    fetch = fetch or _get
    records, start, total = [], 0, None
    while True:
        url = f"{ARXIV_API}?search_query={urllib.parse.quote(query)}&start={start}&max_results={page}&sortBy=submittedDate&sortOrder=descending"
        try:
            batch, total = parse_arxiv_feed(fetch(url))
        except (urllib.error.URLError, ET.ParseError, OSError) as exc:
            return records, f"failed at start={start}: {exc}"
        records.extend(batch)
        start += page
        if not batch or start >= min(total, max_total):
            break
        time.sleep(GAP)
    return records, "ok"


# ---- OpenAlex -----------------------------------------------------------------------------------------------

def abstract_from_inverted(index: dict | None) -> str:
    if not index:
        return ""
    positions = []
    for word, places in index.items():
        for p in places:
            positions.append((p, word))
    return " ".join(w for _, w in sorted(positions))


def parse_openalex_page(payload: dict) -> list[dict]:
    out = []
    for w in payload.get("results", []):
        doi = (w.get("doi") or "").replace("https://doi.org/", "").lower()
        arxiv = ""
        if doi.startswith("10.48550/arxiv."):
            arxiv = doi.split("arxiv.", 1)[1]
        for loc in w.get("locations") or []:
            for u in (loc.get("landing_page_url") or "", loc.get("pdf_url") or ""):
                if not arxiv and "arxiv.org" in u:
                    arxiv = arxiv_id_from_text(u)
        src = (w.get("primary_location") or {}).get("source") or {}
        authors = [(a.get("author") or {}).get("display_name", "") for a in w.get("authorships") or []]
        out.append({
            "openalex": (w.get("id") or "").rsplit("/", 1)[-1],
            "title": " ".join((w.get("title") or w.get("display_name") or "").split()),
            "abstract": abstract_from_inverted(w.get("abstract_inverted_index")),
            "year": w.get("publication_year") or "", "date": w.get("publication_date") or "",
            "venue": src.get("display_name") or "", "type": w.get("type") or "",
            "doi": "" if doi.startswith("10.48550/arxiv.") else doi, "arxiv": arxiv,
            "first_author": authors[0] if authors else "", "authors": authors,
        })
    return out


def openalex_search(text: str, date_from: str, api_key: str = "", max_total: int = 5000, fetch=None) -> tuple[list[dict], str, str]:
    """Title-and-abstract search, page by page (OpenAlex allows up to 10,000 results this way).

    Since February 2026 OpenAlex requires an API key for more than a first page; without one the first page is
    returned and the status says so. The key is never written to the log: the URL is logged without it."""
    first_url = ""
    if OFFLINE and fetch is None:
        return [], "offline", first_url
    fetch = fetch or _get
    records, page, total = [], 1, None
    flt = f"title_and_abstract.search:{text},from_publication_date:{date_from}"
    while True:
        base = (f"{OPENALEX_API}?filter={urllib.parse.quote(flt, safe=':,')}&per-page=100&page={page}"
                f"&select=id,doi,title,display_name,publication_year,publication_date,type,primary_location,locations,authorships,abstract_inverted_index")
        first_url = first_url or base
        url = base + (f"&api_key={api_key}" if api_key else "")
        try:
            payload = json.loads(fetch(url))
        except urllib.error.HTTPError as exc:
            if exc.code == 401 and page > 1:
                return records, f"first page only: a page {page} request was refused (401); an API key is needed for more", first_url
            return records, f"failed after {len(records)}: HTTP Error {exc.code}", first_url
        except (urllib.error.URLError, json.JSONDecodeError, OSError) as exc:
            return records, f"failed after {len(records)}: {exc}", first_url
        results = payload.get("results") or []
        records.extend(parse_openalex_page(payload))
        total = int((payload.get("meta") or {}).get("count") or 0)
        if not results or len(records) >= min(total, max_total):
            break
        if not api_key:
            return records, f"first page only ({len(records)} of {total}): no OpenAlex API key in .secrets/openalex_key.txt", first_url
        page += 1
        time.sleep(1.0)
    return records, "ok", first_url


# ---- DBLP ---------------------------------------------------------------------------------------------------

def parse_dblp_page(payload: dict) -> list[dict]:
    out = []
    hits = ((payload.get("result") or {}).get("hits") or {}).get("hit") or []
    for h in hits:
        info = h.get("info") or {}
        authors = info.get("authors", {}).get("author", []) if isinstance(info.get("authors"), dict) else []
        if isinstance(authors, dict):
            authors = [authors]
        names = [a.get("text", "") if isinstance(a, dict) else str(a) for a in authors]
        ee = info.get("ee") or ""
        out.append({
            "dblp": info.get("key", ""), "title": " ".join((info.get("title") or "").rstrip(".").split()),
            "year": info.get("year", ""), "date": "", "venue": info.get("venue", "") if isinstance(info.get("venue"), str) else " / ".join(info.get("venue") or []),
            "doi": (info.get("doi") or "").lower(), "arxiv": arxiv_id_from_text(ee) if "arxiv" in ee.lower() else "",
            "type": info.get("type", ""), "first_author": names[0] if names else "", "authors": names, "abstract": "",
        })
    return out


def dblp_search(text: str, max_total: int = 3000, fetch=None) -> tuple[list[dict], str, str]:
    first_url = ""
    if OFFLINE and fetch is None:
        return [], "offline", first_url
    fetch = fetch or _get
    records, offset = [], 0
    while True:
        url = f"{DBLP_API}?q={urllib.parse.quote(text)}&h=1000&f={offset}&format=json"
        first_url = first_url or url
        body = b""
        for attempt in range(2):
            try:
                body = fetch(url)
                payload = json.loads(body)
                break
            except json.JSONDecodeError:
                snippet = body[:160].decode("utf-8", "ignore").replace("\n", " ")
                if attempt == 0:
                    time.sleep(10.0)
                    continue
                return records, f"failed after {len(records)}: response is not JSON: {snippet}", first_url
            except (urllib.error.URLError, OSError) as exc:
                if attempt == 0:
                    time.sleep(10.0)
                    continue
                return records, f"failed after {len(records)}: {exc}", first_url
        batch = parse_dblp_page(payload)
        records.extend(batch)
        total = int(((payload.get("result") or {}).get("hits") or {}).get("@total", 0) or 0)
        offset += 1000
        if not batch or offset >= min(total, max_total):
            break
        time.sleep(2.0)
    return records, "ok", first_url


# ---- exported record files (RIS, BibTeX, CSV) ------------------------------------------------------------------

def parse_ris(text: str) -> list[dict]:
    out, cur = [], {}
    for line in text.splitlines():
        m = re.match(r"^([A-Z][A-Z0-9])  - ?(.*)$", line)
        if not m:
            continue
        tag, val = m.group(1), m.group(2).strip()
        if tag == "TY":
            cur = {"authors": []}
        elif tag == "ER":
            if cur:
                out.append(_ris_record(cur))
            cur = {}
        elif tag in ("TI", "T1"):
            cur["title"] = val
        elif tag in ("AU", "A1"):
            cur["authors"].append(val)
        elif tag in ("PY", "Y1"):
            cur["year"] = val[:4]
        elif tag == "DO":
            cur["doi"] = val.lower().replace("https://doi.org/", "")
        elif tag in ("AB", "N2"):
            cur["abstract"] = (cur.get("abstract", "") + " " + val).strip()
        elif tag in ("JO", "T2", "JF", "BT"):
            cur.setdefault("venue", val)
        elif tag == "UR":
            cur.setdefault("url", val)
    return out


def _ris_record(cur: dict) -> dict:
    return {"title": cur.get("title", ""), "first_author": (cur.get("authors") or [""])[0], "authors": cur.get("authors", []),
            "year": cur.get("year", ""), "date": "", "venue": cur.get("venue", ""), "doi": cur.get("doi", ""),
            "arxiv": arxiv_id_from_text(cur.get("url", "")) if "arxiv" in cur.get("url", "") else "", "abstract": cur.get("abstract", "")}


def parse_bibtex(text: str) -> list[dict]:
    out = []
    for entry in re.split(r"\n@", "\n" + text)[1:]:
        fields = dict((k.lower(), re.sub(r"\s+", " ", v.strip().strip("{}\",")))
                      for k, v in re.findall(r"(\w+)\s*=\s*[{\"]((?:[^{}]|\{[^{}]*\})*)[}\"]", entry))
        authors = [a.strip() for a in re.split(r"\s+and\s+", fields.get("author", "")) if a.strip()]
        out.append({"title": fields.get("title", ""), "first_author": authors[0] if authors else "", "authors": authors,
                    "year": fields.get("year", ""), "date": "", "venue": fields.get("journal") or fields.get("booktitle") or "",
                    "doi": fields.get("doi", "").lower(), "arxiv": arxiv_id_from_text(fields.get("eprint", "") or fields.get("url", "")),
                    "abstract": fields.get("abstract", "")})
    return out


def parse_export_file(path: Path) -> list[dict]:
    path = Path(path)
    text = path.read_text(encoding="utf-8", errors="ignore")
    if path.suffix.lower() == ".ris":
        return parse_ris(text)
    if path.suffix.lower() in (".bib", ".bibtex"):
        return parse_bibtex(text)
    if path.suffix.lower() == ".csv":
        rows = list(csv.DictReader(text.splitlines()))
        out = []
        for r in rows:
            low = {k.lower().strip(): (v or "") for k, v in r.items() if k}
            title = low.get("title") or low.get("document title") or low.get("item title") or ""
            out.append({"title": title, "first_author": (low.get("authors") or low.get("author") or "").split(";")[0].strip(),
                        "year": (low.get("year") or low.get("publication year") or "")[:4], "date": "",
                        "venue": low.get("publication title") or low.get("source title") or low.get("venue") or "",
                        "doi": (low.get("doi") or "").lower(), "arxiv": arxiv_id_from_text(low.get("url", "") or low.get("pdf link", "")),
                        "abstract": low.get("abstract") or ""})
        return out
    return []


# ---- citation lists from PDFs ----------------------------------------------------------------------------------

def arxiv_ids_in_pdf(path: Path) -> list[str]:
    from pypdf import PdfReader
    text = " ".join((p.extract_text() or "") for p in PdfReader(str(path)).pages)
    text = re.sub(r"\s+", " ", text)
    return sorted(set(m.group(1) for m in ARXIV_ID_RE.finditer(text)))


def arxiv_metadata_for_ids(ids: list[str], cache_path: Path, batch: int = 50, fetch=None) -> dict:
    cache_path = Path(cache_path)
    cache = json.loads(cache_path.read_text()) if cache_path.exists() else {}
    missing = [i for i in ids if i not in cache]
    if (OFFLINE and fetch is None) or not missing:
        return cache
    fetch = fetch or _get
    for s in range(0, len(missing), batch):
        chunk = missing[s:s + batch]
        url = f"{ARXIV_API}?id_list={','.join(chunk)}&max_results={len(chunk)}"
        try:
            recs, _ = parse_arxiv_feed(fetch(url))
        except (urllib.error.URLError, ET.ParseError, OSError) as exc:
            print("arXiv id_list request failed:", exc)
            break
        for r in recs:
            cache[r["arxiv"]] = r
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        cache_path.write_text(json.dumps(cache, ensure_ascii=False))
        time.sleep(GAP)
    return cache


# ---- merging and screening ------------------------------------------------------------------------------------

def record_key(rec: dict) -> str:
    if rec.get("arxiv"):
        return f"arxiv:{rec['arxiv']}"
    if rec.get("doi"):
        return f"doi:{rec['doi']}"
    return f"title:{norm_key(rec.get('title', ''))}"


def merge_hits(hits: list[tuple[str, str, dict]]) -> list[dict]:
    """hits: (source, query_id, record). One row per key; sources and queries aggregated; preprint and published
    versions of one title linked through linked_to (the arXiv record is the root when present)."""
    merged: dict[str, dict] = {}
    for source, qid, rec in hits:
        key = record_key(rec)
        row = merged.get(key)
        if row is None:
            row = {"key": key, "title": rec.get("title", ""), "first_author": rec.get("first_author", ""),
                   "year": str(rec.get("year") or (rec.get("date") or "")[:4]), "date": rec.get("date", ""),
                   "venue": rec.get("venue", ""), "arxiv": rec.get("arxiv", ""), "arxiv_version": rec.get("arxiv_version", ""),
                   "doi": rec.get("doi", ""), "abstract": rec.get("abstract", ""), "_sources": set(), "_queries": set(),
                   "n_hits": 0, "linked_to": ""}
            merged[key] = row
        for f in ("title", "first_author", "venue", "doi", "abstract", "date", "arxiv_version"):
            if not row.get(f) and rec.get(f):
                row[f] = rec[f]
        if not row.get("year") and (rec.get("year") or rec.get("date")):
            row["year"] = str(rec.get("year") or rec["date"][:4])
        row["_sources"].add(source)
        row["_queries"].add(qid)
        row["n_hits"] += 1
    by_title: dict[str, str] = {}
    for key, row in merged.items():
        t = norm_key(row["title"])
        if not t:
            continue
        root = by_title.get(t)
        if root is None:
            by_title[t] = key
        else:
            # prefer the arXiv record as the root so the preprint and the published article count once
            if key.startswith("arxiv:") and not root.startswith("arxiv:"):
                merged[root]["linked_to"] = key
                by_title[t] = key
            else:
                row["linked_to"] = root
    rows = []
    for row in merged.values():
        row["sources"] = ";".join(sorted(row.pop("_sources")))
        row["queries"] = ";".join(sorted(row.pop("_queries")))
        rows.append(row)
    rows.sort(key=lambda r: (r["linked_to"] != "", -(r["n_hits"]), r["key"]))
    return rows


def apply_decisions(rows: list[dict], decisions: list[dict]) -> tuple[list[dict], list[str]]:
    """Attach decision, reason, access_status and note by key; report keys with invalid values or no match."""
    problems = []
    by_key = {d["key"]: d for d in decisions}
    for row in rows:
        d = by_key.get(row["key"], {})
        row["decision"] = d.get("decision", "pending")
        row["reason"] = d.get("reason", "")
        row["access_status"] = d.get("access_status", "")
        row["note"] = d.get("note", "")
        if d and d.get("decision") not in DECISIONS:
            problems.append(f"{row['key']}: decision must be one of {sorted(DECISIONS)}")
        if d.get("decision") == "exclude" and d.get("reason") not in EXCLUSION_REASONS:
            problems.append(f"{row['key']}: exclusion needs one reason from the fixed list")
    unknown = [k for k in by_key if k not in {r["key"] for r in rows}]
    problems += [f"decision for a key not in the screening table: {k}" for k in unknown]
    return rows, problems


def prisma_counts(log_rows: list[dict], rows: list[dict]) -> dict:
    per_source: dict[str, int] = {}
    for r in log_rows:
        if str(r.get("count", "")).isdigit():
            per_source[r["source"]] = per_source.get(r["source"], 0) + int(r["count"])
    roots = [r for r in rows if not r.get("linked_to")]
    out = {"records_identified_per_source": per_source,
           "records_identified_total": sum(per_source.values()),
           "unique_records_after_dedup": len(rows),
           "studies_after_version_linking": len(roots),
           "screened": sum(1 for r in roots if r.get("decision") not in ("", "pending", None)),
           "pending": sum(1 for r in roots if r.get("decision") in ("", "pending", None)),
           "included": sum(1 for r in roots if r.get("decision") == "include"),
           "unsure_to_full_text": sum(1 for r in roots if r.get("decision") == "unsure"),
           "excluded_by_reason": {}}
    for r in roots:
        if r.get("decision") == "exclude":
            out["excluded_by_reason"][r.get("reason", "")] = out["excluded_by_reason"].get(r.get("reason", ""), 0) + 1
    return out

In [ ]:
%%writefile tests/test_search.py
"""Parsers, merging and the screening-decision checks of the search module."""
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
import search  # noqa: E402

ARXIV_FEED = b"""<?xml version="1.0" encoding="UTF-8"?>
<feed xmlns="http://www.w3.org/2005/Atom" xmlns:arxiv="http://arxiv.org/schemas/atom" xmlns:opensearch="http://a9.com/-/spec/opensearch/1.1/">
  <opensearch:totalResults>2</opensearch:totalResults>
  <entry>
    <id>http://arxiv.org/abs/2402.06363v2</id>
    <published>2024-02-09T00:00:00Z</published><updated>2024-05-01T00:00:00Z</updated>
    <title>StruQ: Defending Against Prompt Injection
  with Structured Queries</title>
    <summary>We propose structured queries.</summary>
    <author><name>Sizhe Chen</name></author><author><name>Other Author</name></author>
    <arxiv:primary_category term="cs.CR"/>
  </entry>
  <entry>
    <id>http://arxiv.org/abs/2403.14720v1</id>
    <published>2024-03-21T00:00:00Z</published><updated>2024-03-21T00:00:00Z</updated>
    <title>Defending Against Indirect Prompt Injection Attacks With Spotlighting</title>
    <summary>Spotlighting.</summary>
    <author><name>Keegan Hines</name></author>
    <arxiv:journal_ref>Some venue 2024</arxiv:journal_ref>
    <arxiv:doi>10.1000/EXAMPLE</arxiv:doi>
  </entry>
</feed>"""

OPENALEX_PAGE = {
    "meta": {"next_cursor": None},
    "results": [
        {"id": "https://openalex.org/W1", "doi": "https://doi.org/10.48550/arXiv.2402.06363", "title": "StruQ: Defending Against Prompt Injection with Structured Queries",
         "publication_year": 2024, "publication_date": "2024-02-09", "type": "preprint",
         "primary_location": {"source": {"display_name": "arXiv (Cornell University)"}}, "locations": [],
         "authorships": [{"author": {"display_name": "Sizhe Chen"}}],
         "abstract_inverted_index": {"We": [0], "propose": [1], "structured": [2], "queries.": [3]}},
        {"id": "https://openalex.org/W2", "doi": "https://doi.org/10.1109/SP.2025.1", "title": "StruQ: Defending Against Prompt Injection with Structured Queries",
         "publication_year": 2025, "publication_date": "2025-05-01", "type": "article",
         "primary_location": {"source": {"display_name": "USENIX Security"}}, "locations": [{"landing_page_url": "https://www.usenix.org/x"}],
         "authorships": [{"author": {"display_name": "Sizhe Chen"}}], "abstract_inverted_index": None},
    ],
}

DBLP_PAGE = {"result": {"hits": {"@total": "1", "hit": [
    {"info": {"key": "conf/uss/Chen25", "title": "StruQ: Defending Against Prompt Injection with Structured Queries.",
              "venue": "USENIX Security Symposium", "year": "2025", "doi": "10.1109/SP.2025.1", "ee": "https://doi.org/10.1109/SP.2025.1",
              "type": "Conference and Workshop Papers", "authors": {"author": [{"text": "Sizhe Chen"}, {"text": "Other"}]}}}]}}}


def test_parse_arxiv_feed_reads_ids_versions_titles_and_total():
    recs, total = search.parse_arxiv_feed(ARXIV_FEED)
    assert total == 2 and [r["arxiv"] for r in recs] == ["2402.06363", "2403.14720"]
    assert recs[0]["arxiv_version"] == 2 and recs[0]["title"].endswith("with Structured Queries")
    assert recs[0]["first_author"] == "Sizhe Chen" and recs[0]["primary_category"] == "cs.CR"
    assert recs[1]["doi"] == "10.1000/example" and recs[1]["venue"] == "Some venue 2024"


def test_arxiv_query_string_quotes_phrases_and_adds_window():
    q = search.arxiv_query_string(["prompt injection", "defense"], "20230101", "20261120")
    assert q.startswith('abs:"prompt injection" AND abs:defense AND (cat:cs.CR OR cat:cs.CL OR cat:cs.LG OR cat:cs.AI)')
    assert q.endswith("submittedDate:[202301010000 TO 202611202359]")


def test_parse_openalex_page_recovers_arxiv_id_and_abstract():
    recs = search.parse_openalex_page(OPENALEX_PAGE)
    assert recs[0]["arxiv"] == "2402.06363" and recs[0]["doi"] == "" and recs[0]["abstract"] == "We propose structured queries."
    assert recs[1]["doi"] == "10.1109/sp.2025.1" and recs[1]["venue"] == "USENIX Security" and recs[1]["abstract"] == ""


def test_parse_dblp_page():
    recs = search.parse_dblp_page(DBLP_PAGE)
    assert recs[0]["title"] == "StruQ: Defending Against Prompt Injection with Structured Queries"
    assert recs[0]["doi"] == "10.1109/sp.2025.1" and recs[0]["first_author"] == "Sizhe Chen" and recs[0]["year"] == "2025"


def test_arxiv_id_regex_accepts_only_2023_to_2026_ids():
    assert search.arxiv_id_from_text("see arXiv:2402.06363v2 for details") == "2402.06363"
    assert search.arxiv_id_from_text("arXiv preprint arXiv:2106.09685") == ""
    assert search.arxiv_id_from_text("doi 10.1234/5678.91011") == ""


def test_merge_hits_dedupes_across_sources_and_links_versions():
    a, _ = search.parse_arxiv_feed(ARXIV_FEED)
    o = search.parse_openalex_page(OPENALEX_PAGE)
    d = search.parse_dblp_page(DBLP_PAGE)
    hits = [("arxiv", "q01", a[0]), ("arxiv", "q05", a[0]), ("arxiv", "q01", a[1]),
            ("openalex", "o01", o[0]), ("openalex", "o01", o[1]), ("dblp", "d01", d[0])]
    rows = search.merge_hits(hits)
    by_key = {r["key"]: r for r in rows}
    struq = by_key["arxiv:2402.06363"]
    assert struq["n_hits"] == 3 and struq["sources"] == "arxiv;openalex" and struq["queries"] == "o01;q01;q05"
    published = by_key["doi:10.1109/sp.2025.1"]
    assert published["linked_to"] == "arxiv:2402.06363" and published["sources"] == "dblp;openalex"
    assert by_key["arxiv:2403.14720"]["linked_to"] == ""
    assert rows[0]["linked_to"] == "" and rows[-1]["linked_to"] != ""


def test_apply_decisions_flags_bad_values_and_unknown_keys():
    rows = [{"key": "arxiv:1"}, {"key": "arxiv:2"}, {"key": "arxiv:3"}]
    decisions = [{"key": "arxiv:1", "decision": "include"}, {"key": "arxiv:2", "decision": "exclude", "reason": "made up"},
                 {"key": "arxiv:9", "decision": "exclude", "reason": "attack only"}]
    rows, problems = search.apply_decisions(rows, decisions)
    assert rows[0]["decision"] == "include" and rows[2]["decision"] == "pending"
    assert any("arxiv:2" in p for p in problems) and any("arxiv:9" in p for p in problems)


def test_prisma_counts():
    log = [{"source": "arxiv", "count": "10"}, {"source": "arxiv", "count": "5"}, {"source": "dblp", "count": "offline"}]
    rows = [{"key": "a", "linked_to": "", "decision": "include"}, {"key": "b", "linked_to": "a", "decision": "include"},
            {"key": "c", "linked_to": "", "decision": "exclude", "reason": "attack only"}, {"key": "d", "linked_to": "", "decision": "pending"}]
    c = search.prisma_counts(log, rows)
    assert c["records_identified_per_source"] == {"arxiv": 15} and c["unique_records_after_dedup"] == 4
    assert c["studies_after_version_linking"] == 3 and c["included"] == 1 and c["pending"] == 1
    assert c["excluded_by_reason"] == {"attack only": 1}


def test_parse_ris_and_bibtex():
    ris = "TY  - JOUR\nTI  - A Title\nAU  - Doe, J.\nAU  - Roe, R.\nPY  - 2025\nDO  - https://doi.org/10.1/ABC\nAB  - Part one.\nAB  - Part two.\nER  - \n"
    r = search.parse_ris(ris)[0]
    assert r["title"] == "A Title" and r["first_author"] == "Doe, J." and r["doi"] == "10.1/abc" and r["abstract"] == "Part one. Part two."
    bib = '@inproceedings{x,\n title={Another {Title}},\n author={Jane Doe and John Roe},\n year={2024},\n booktitle={Proc. X},\n eprint={2402.06363}\n}\n'
    b = search.parse_bibtex(bib)[0]
    assert b["first_author"] == "Jane Doe" and b["venue"] == "Proc. X" and b["arxiv"] == "2402.06363" and b["year"] == "2024"


def test_search_functions_report_failure_instead_of_raising():
    def boom(url, timeout=90):
        raise OSError("HTTP Error 503")
    recs, status = search.arxiv_search("abs:x", fetch=boom)
    assert recs == [] and status.startswith("failed")
    recs, status, _ = search.openalex_search("x", "2023-01-01", fetch=boom)
    assert recs == [] and status.startswith("failed")
    recs, status, _ = search.dblp_search("x", fetch=boom)
    assert recs == [] and status.startswith("failed")


def test_search_functions_parse_fixture_responses():
    calls = []
    def fake(url, timeout=90):
        calls.append(url)
        if url.startswith(search.ARXIV_API):
            return ARXIV_FEED
        if url.startswith(search.OPENALEX_API):
            return json.dumps(OPENALEX_PAGE).encode()
        return json.dumps(DBLP_PAGE).encode()
    recs, status = search.arxiv_search('abs:"prompt injection"', fetch=fake)
    assert status == "ok" and len(recs) == 2
    recs, status, url = search.openalex_search('"prompt injection"', "2023-01-01", api_key="k", fetch=fake)
    assert status == "ok" and len(recs) == 2 and "title_and_abstract.search" in url and "api_key" not in url
    recs, status, url = search.dblp_search("prompt injection", fetch=fake)
    assert status == "ok" and len(recs) == 1 and "format=json" in url


def test_openalex_without_key_stops_after_the_first_page_and_says_so():
    big = {"meta": {"count": 350}, "results": OPENALEX_PAGE["results"]}
    recs, status, _ = search.openalex_search("x", "2023-01-01", fetch=lambda url, timeout=90: json.dumps(big).encode())
    assert len(recs) == 2 and status.startswith("first page only") and "API key" in status


def test_openalex_401_on_a_later_page_is_reported_not_raised():
    import urllib.error
    big = {"meta": {"count": 350}, "results": OPENALEX_PAGE["results"]}
    def fake(url, timeout=90):
        if "page=1&" in url or url.endswith("page=1"):
            return json.dumps(big).encode()
        raise urllib.error.HTTPError(url, 401, "Unauthorized", None, None)
    recs, status, _ = search.openalex_search("x", "2023-01-01", api_key="bad", fetch=fake)
    assert len(recs) == 2 and "401" in status


def test_dblp_reports_non_json_body():
    search_gap = search.time.sleep
    search.time.sleep = lambda s: None
    try:
        recs, status, _ = search.dblp_search("x", fetch=lambda url, timeout=90: b"<html>Too Many Requests</html>")
    finally:
        search.time.sleep = search_gap
    assert recs == [] and "not JSON" in status and "Too Many Requests" in status

In [ ]:
%%writefile schema/search_queries.yaml
# Search strings committed before the search (plan v2, A5). Each arXiv query requires every term in the abstract,
# within cs.CR, cs.CL, cs.LG or cs.AI, submitted from 2023-01-01 to the run date. OpenAlex is given the same terms
# as a title-and-abstract search; DBLP, which indexes titles only, gets the three broad strings.
window_start: "2023-01-01"
arxiv_categories: [cs.CR, cs.CL, cs.LG, cs.AI]
arxiv_queries:
  - {id: q01, must: ["prompt injection", "defense"]}
  - {id: q02, must: ["prompt injection", "defence"]}
  - {id: q03, must: ["prompt injection", "defenses"]}
  - {id: q04, must: ["prompt injection", "defending"]}
  - {id: q05, must: ["prompt injection", "fine-tuning"]}
  - {id: q06, must: ["prompt injection", "preference optimization"]}
  - {id: q07, must: ["prompt injection", "distillation"]}
  - {id: q08, must: ["prompt injection", "instruction hierarchy"]}
  - {id: q09, must: ["prompt injection", "segment embedding"]}
  - {id: q10, must: ["prompt injection", "delimiter"]}
  - {id: q11, must: ["prompt injection", "spotlighting"]}
  - {id: q12, must: ["prompt injection", "steering"]}
  - {id: q13, must: ["prompt injection", "attention"]}
  - {id: q14, must: ["prompt injection", "representation"]}
  - {id: q15, must: ["prompt injection", "position"]}
  - {id: q16, must: ["prompt injection", "soft token"]}
  - {id: q17, must: ["instruction-data separation"]}
  - {id: q18, must: ["instruction data separation"]}
  - {id: q19, must: ["indirect prompt injection", "defense"]}
  - {id: q20, must: ["tool output", "injection", "defense"]}
openalex_queries:
  - {id: o01, text: '"prompt injection" AND (defense OR defence OR defending OR defenses)'}
  - {id: o02, text: '"prompt injection" AND (fine-tuning OR "preference optimization" OR distillation)'}
  - {id: o03, text: '"prompt injection" AND ("instruction hierarchy" OR "segment embedding" OR delimiter OR spotlighting)'}
  - {id: o04, text: '"prompt injection" AND (steering OR attention OR representation OR position OR "soft token")'}
  - {id: o05, text: '"instruction-data separation" OR "instruction data separation"'}
  - {id: o06, text: '"indirect prompt injection" AND defense'}
  - {id: o07, text: '"tool output" AND injection AND defense'}
dblp_queries:
  - {id: d01, text: "prompt injection"}
  - {id: d02, text: "instruction data separation"}
  - {id: d03, text: "indirect prompt injection"}
# Seed defences from the plan's provisional list (A5). Each name is searched on arXiv in the title or abstract
# together with "prompt injection" in the abstract; known arXiv ids are fetched directly. Hits are logged under
# the source "seed list".
seed_names: ["Spotlighting", "Signed-Prompt", "FATH", "Mixture of Encodings", "Polymorphic Prompt", "Prompt Fencing",
             "BIPIA", "StruQ", "SecAlign", "Meta SecAlign", "Instruction Hierarchy", "Jatmo",
             "Instructional Segment Embedding", "DefensiveTokens", "ASIDE", "DRIP", "CachePrune", "CounterSteer",
             "Heads of Security", "ICON", "AGRI", "V-Steer", "SecOPD", "RAISED", "twin distillation", "RETA", "COPA",
             "LocalAlign", "Instruction Ladder", "Control Illusion", "Position, Role or Label"]
seed_arxiv_ids: ["2505.00626", "2505.18333", "2510.09023", "2601.07185", "2606.30783", "2610.33201"]
# Backward citation searching: every arXiv identifier printed in these PDFs (data/surveys/<id>.pdf) is collected,
# its metadata fetched, and the record logged under the source "citations" with the citing paper as the query id.
citation_sources: ["hong_2510.15476", "ji_2511.15203", "geng_cmc_2026", "wang_2602.10453", "sammartino_kbs_2026",
                   "correia_2601.22240", "hashemi_2606.31639", "mourad_electronics_2026", "secfid_2606.30783",
                   "jia_2505.18333", "nasr_2510.09023", "li_s_2601.07185", "deep_2604.23887"]

## Stage 2. arXiv queries

In [ ]:
import json, time, yaml, datetime as dt
import search, corpus, paths
importlib.reload(search); importlib.reload(corpus); importlib.reload(paths)
paths.ensure_dirs()

Q = yaml.safe_load(Path("schema/search_queries.yaml").read_text())
RUN_DATE = dt.date.today().isoformat()
DATE_FROM, DATE_TO = Q["window_start"].replace("-", ""), RUN_DATE.replace("-", "")
RAW = paths.REPORTS / "search_raw"; RAW.mkdir(parents=True, exist_ok=True)
log_rows, hits = [], []

def slim(recs):
    return [{k: r.get(k, "") for k in ("arxiv", "doi", "title", "date", "year", "venue")} for r in recs]

def log(source, qid, sent, fields, filters, recs, status, file):
    log_rows.append({"date": RUN_DATE, "source": source, "query_id": qid, "query_as_sent": sent, "fields": fields,
                     "filters": filters, "count": len(recs), "status": status, "file": file})
    print(f"{source:<10} {qid:<6} {len(recs):>5}  {status}")

for q in Q["arxiv_queries"]:
    qs = search.arxiv_query_string(q["must"], DATE_FROM, DATE_TO, Q["arxiv_categories"])
    recs, status = search.arxiv_search(qs)
    f = RAW / f"arxiv_{q['id']}.json"; f.write_text(json.dumps(slim(recs), ensure_ascii=False))
    log("arxiv", q["id"], qs, "abs", f"categories {Q['arxiv_categories']}; submitted {Q['window_start']} to {RUN_DATE}", recs, status, str(f.relative_to(paths.REPO_ROOT)))
    hits += [("arxiv", q["id"], r) for r in recs]
    time.sleep(0 if search.OFFLINE else search.GAP)          # arXiv asks for one request every three seconds
print("arXiv hits so far:", len(hits), "| offline" if search.OFFLINE else "")

## Stage 3. OpenAlex and DBLP queries

In [ ]:
for q in Q["openalex_queries"]:
    recs, status, url = search.openalex_search(q["text"], Q["window_start"], api_key=OPENALEX_KEY)
    f = RAW / f"openalex_{q['id']}.json"; f.write_text(json.dumps(slim(recs), ensure_ascii=False))
    log("openalex", q["id"], url or q["text"], "title_and_abstract", f"from_publication_date {Q['window_start']}", recs, status, str(f.relative_to(paths.REPO_ROOT)))
    hits += [("openalex", q["id"], r) for r in recs]
for q in Q["dblp_queries"]:
    recs, status, url = search.dblp_search(q["text"])
    recs = [r for r in recs if str(r.get("year", "")).isdigit() and int(r["year"]) >= int(Q["window_start"][:4])]
    f = RAW / f"dblp_{q['id']}.json"; f.write_text(json.dumps(slim(recs), ensure_ascii=False))
    log("dblp", q["id"], url or q["text"], "title", f"year >= {Q['window_start'][:4]} applied after retrieval", recs, status, str(f.relative_to(paths.REPO_ROOT)))
    hits += [("dblp", q["id"], r) for r in recs]
print("hits so far:", len(hits))

## Stage 4. Exported record files from browser searches, if any

In [ ]:
EXPORTS = paths.DATA / "search_exports"
files = sorted(p for p in EXPORTS.rglob("*") if p.suffix.lower() in (".ris", ".bib", ".bibtex", ".csv")) if EXPORTS.exists() else []
for p in files:
    source = p.parent.name if p.parent != EXPORTS else "export"
    recs = search.parse_export_file(p)
    log(source, p.stem, f"imported file {p.name}", "as exported", "as exported", recs, "imported", str(p.relative_to(paths.REPO_ROOT)))
    hits += [(source, p.stem, r) for r in recs]
print("export files imported:", len(files), "| hits so far:", len(hits))

## Stage 5. Seed defences by name and citation lists of the closest papers

In [ ]:
seed_cache = paths.REPORTS / "arxiv_meta_seeds.json"
for name in Q["seed_names"]:
    qs = f'(ti:"{name}" OR abs:"{name}") AND abs:"prompt injection" AND submittedDate:[{DATE_FROM}0000 TO {DATE_TO}2359]'
    recs, status = search.arxiv_search(qs, max_total=200)
    log("seed list", f"name:{name}", qs, "ti, abs", f"submitted {Q['window_start']} to {RUN_DATE}", recs, status, "")
    hits += [("seed list", f"name:{name}", r) for r in recs]
    time.sleep(0 if search.OFFLINE else search.GAP)
meta = search.arxiv_metadata_for_ids(Q["seed_arxiv_ids"], seed_cache)
seed_recs = [meta[i] for i in Q["seed_arxiv_ids"] if i in meta]
log("seed list", "ids", ",".join(Q["seed_arxiv_ids"]), "id_list", "", seed_recs, "ok" if seed_recs or search.OFFLINE else "none fetched", str(seed_cache.relative_to(paths.REPO_ROOT)))
hits += [("seed list", "ids", r) for r in seed_recs]

cite_cache = paths.REPORTS / "arxiv_meta_citations.json"
cited: dict[str, set] = {}
for src_id in Q["citation_sources"]:
    pdf = paths.DATA_SURVEYS / f"{src_id}.pdf"
    if not pdf.exists():
        log("citations", f"cite:{src_id}", "PDF not on disk", "reference list", "arXiv ids 2023 onwards", [], "PDF missing", "")
        continue
    ids = [i for i in search.arxiv_ids_in_pdf(pdf) if i not in src_id]
    cited[src_id] = set(ids)
all_cited = sorted(set().union(*cited.values())) if cited else []
cmeta = search.arxiv_metadata_for_ids(all_cited, cite_cache)
for src_id, ids in cited.items():
    recs = [cmeta[i] for i in sorted(ids) if i in cmeta]
    st = "ok" if recs else ("no arXiv ids from 2023 onwards in the PDF text" if not ids else ("offline" if search.OFFLINE else "ids found, no metadata"))
    log("citations", f"cite:{src_id}", f"{len(ids)} arXiv ids printed in the PDF", "reference list", "arXiv ids 2023 onwards; metadata by id_list",
        recs, st, str(cite_cache.relative_to(paths.REPO_ROOT)))
    hits += [("citations", f"cite:{src_id}", r) for r in recs]
print("cited arXiv ids (2023 onwards) across the citation sources:", len(all_cited), "| hits total:", len(hits))

## Stage 6. Merge, apply screening decisions, PRISMA counts

In [ ]:
rows = search.merge_hits(hits)
decisions = search.read_csv(Path("schema/screening_decisions.csv"))
previous = {r["key"]: r for r in search.read_csv(paths.REPORTS / "screening.csv") if r.get("decision") not in ("", "pending")}
for key, prev in previous.items():                       # decisions already applied earlier survive if the file lacks them
    if key not in {d["key"] for d in decisions}:
        decisions.append({"key": key, "decision": prev["decision"], "reason": prev.get("reason", ""),
                          "access_status": prev.get("access_status", ""), "note": prev.get("note", "")})
rows, problems = search.apply_decisions(rows, decisions)
search.write_csv(paths.REPORTS / "screening.csv", rows, search.SCREEN_FIELDS)

old_log = [r for r in search.read_csv(paths.REPORTS / "search_log.csv") if r.get("date") != RUN_DATE]
search.write_csv(paths.REPORTS / "search_log.csv", old_log + log_rows, search.LOG_FIELDS)

counts = search.prisma_counts(log_rows, rows)
(paths.REPORTS / "prisma_counts.json").write_text(json.dumps(counts, indent=1))
status = ("CORPUS FROZEN" if counts["pending"] == 0 and counts["screened"] > 0 else
          "SEARCH DONE; SCREENING PENDING" if counts["unique_records_after_dedup"] else "NO RECORDS (offline or all queries failed)")
lines = [f"decision_02: {status}", f"date: {RUN_DATE}",
         f"records identified: {counts['records_identified_total']} across {len(counts['records_identified_per_source'])} sources",
         f"unique records: {counts['unique_records_after_dedup']}; studies after version linking: {counts['studies_after_version_linking']}",
         f"screened: {counts['screened']}; pending: {counts['pending']}; included: {counts['included']}; unsure: {counts['unsure_to_full_text']}",
         "excluded by reason: " + json.dumps(counts["excluded_by_reason"])] + [f"- problem: {p}" for p in problems]
(paths.REPORTS / "decision_02.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
print("per source:", counts["records_identified_per_source"])

## Stage 7. Defence manifest and PDFs (only once decisions exist)

In [ ]:
import re
chosen = [r for r in rows if r.get("decision") in ("include", "unsure") and not r.get("linked_to")]
manifest, status_counts = [], {}
for r in chosen:
    slug = r["arxiv"] if r["arxiv"] else re.sub(r"[^a-z0-9]+", "_", (r["first_author"].split(",")[0].split()[-1] if r["first_author"] else "paper").lower()) + "_" + str(r.get("year", ""))
    dest = paths.DATA_DEFENCES / f"{slug}.pdf"
    if r["arxiv"]:
        url = f"https://arxiv.org/pdf/{r['arxiv']}" + (f"v{r['arxiv_version']}" if r.get("arxiv_version") else "")
        st = corpus.download_pdf(url, dest)
    else:
        st = "present (hand-placed)" if dest.exists() else "missing: place the PDF by hand"
    m = dict(r); m.update({"slug": slug, "pdf_path": str(dest.relative_to(paths.REPO_ROOT)) if dest.exists() else "", "pdf_status": st,
                           "sha256": corpus.sha256(dest) if dest.exists() else "", "pages": corpus.page_count(dest) if dest.exists() else ""})
    manifest.append(m); status_counts[st.split(":")[0]] = status_counts.get(st.split(":")[0], 0) + 1
fields = ["slug", "key", "decision", "title", "first_author", "year", "venue", "arxiv", "arxiv_version", "doi", "pdf_path", "pdf_status", "sha256", "pages", "note"]
if manifest:
    search.write_csv(paths.REPORTS / "defences.csv", manifest, fields)
    print("manifest rows:", len(manifest), status_counts)
else:
    print("manifest deferred: no included or unsure records yet (screening decisions not in place)")
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 8. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "02 search and screen: committed queries run on arXiv, OpenAlex and DBLP, seed and citation records, merged screening table, PRISMA counts"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)